# Notebook to compute results reported in the thesis

1. Per-epoch mIoU comparison between baseline and REPA-Seg runs.
2. Computation of the epoch from which the mIoU difference remains above a specified threshold (+1 percentage point).
3. Summary tables, including LaTeX-formatted output.
4. Visualization of validation mIoU per epoch throughout training for the main experiments.

## 1. Compute last delta miou 
- this computes the delta betweeen the last epochs of the baseline repa run selected in the config globals
- also it computes the epoch, from which delta stays above at least 1 percentage point and onward.

In [16]:
from pathlib import Path
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
from dotenv import load_dotenv

load_dotenv()

REPO_ROOT = Path(os.environ["REPO_ROOT"])
DATA_ROOT = Path(os.environ["DATA_ROOT"])
RUNS_ROOT = Path(os.environ["RUNS_ROOT"])

In [17]:
def get_current_tc(path_name):
    match_obj = re.search(r"(?:^|_)tc(?P<tc>\d+)(?:_|$)", path_name) 
    if match_obj:
        return int(match_obj.group("tc"))
    else:
        return match_obj

def find_metrics(tc):
    matches = []
    for metrics in RUNS_ROOT.rglob("metrics.json"):
        exp_name = metrics.parent.parent.name
        if get_current_tc(exp_name) == tc:
            matches.append(metrics)
    return matches[0]

def load_miou(tc):
    path = find_metrics(tc)
    with open(path) as file:
        metrics = json.load(file)
    rows = []

    for row in metrics.get("epochs", []):
        if row.get("mean_iou") is None:
            continue
        rows.append({
            "epoch": int(row["epoch"]) + 1,
            "miou": float(row["mean_iou"]),
        })
    return pd.DataFrame(rows), path

In [18]:
def print_table(baseline, repa, tc_baseline, tc_repa):
    # compute when it stays above 1 percentage point
    merged = baseline.merge(repa, on="epoch", suffixes=("_baseline", "_repa"),)

    # REPA - baseline for every epoch
    merged["delta"] = (merged["miou_repa"] - merged["miou_baseline"] )

    threshold = 0.01  # 1 percentage point
    stable_from = None

    for i in range(len(merged)):
        if (merged.loc[i:, "delta"] >= threshold).all():
            stable_from = int(merged.loc[i, "epoch"])
            break

    print(f"Delta mIoU stays above 1 pp from epoch index {stable_from+1} onward")

    # find best epoch for both models
    baseline_best_idx = baseline["miou"].idxmax()
    repa_best_idx = repa["miou"].idxmax()

    baseline_best_epoch = int(baseline.loc[baseline_best_idx, "epoch"])
    repa_best_epoch = int(repa.loc[repa_best_idx, "epoch"])

    baseline_best = float( baseline.loc[baseline_best_idx, "miou"])
    repa_best = float(repa.loc[repa_best_idx, "miou"])

    best_delta = repa_best - baseline_best

    print("\nDelta between best epoch checkpoints:")
    print(
        f"tc{tc_baseline}: {baseline_best:.5f} "
        f"at epoch index {baseline_best_epoch+1}"
    )
    print(
        f"tc{tc_repa}: {repa_best:.5f} "
        f"at epoch index {repa_best_epoch+1}"
    )
    print(f"Delta:{best_delta:+.5f}")
    print(f"Delta percentage pts: {100 * best_delta:+.2f} pp")

In [19]:
# Scratch ViT-14/S baseline (tc202) vs. REPA-Seg with DINOv2-B/14 Teacher (tc203)
TC_BASELINE = 202
TC_REPA = 203

print("Scratch ViT-14/S baseline (tc202) vs. REPA-Seg with DINOv2-B/14 Teacher (tc203)\n")
baseline, _ = load_miou(TC_BASELINE)
repa, _ = load_miou(TC_REPA)

matching_epochs = sorted(set(baseline) & set(repa))

print_table(baseline=baseline, repa=repa, tc_baseline=TC_BASELINE, tc_repa=TC_REPA)

Scratch ViT-14/S baseline (tc202) vs. REPA-Seg with DINOv2-B/14 Teacher (tc203)

Delta mIoU stays above 1 pp from epoch index 58 onward

Delta between best epoch checkpoints:
tc202: 0.24216 at epoch index 196
tc203: 0.27308 at epoch index 201
Delta:+0.03092
Delta percentage pts: +3.09 pp


In [20]:
# Scratch ViT-14/S baseline (tc202) vs. REPA-Seg with MAE-B/16 Teacher (tc205)
TC_BASELINE = 202
TC_REPA = 205

print("Scratch ViT-14/S baseline (tc202) vs. REPA-Seg with MAE-B/16 Teacher (tc205)\n")

baseline, _ = load_miou(TC_BASELINE)
repa, _ = load_miou(TC_REPA)

matching_epochs = sorted(set(baseline) & set(repa))

print_table( baseline=baseline, repa=repa, tc_baseline=TC_BASELINE, tc_repa=TC_REPA)

Scratch ViT-14/S baseline (tc202) vs. REPA-Seg with MAE-B/16 Teacher (tc205)



Delta mIoU stays above 1 pp from epoch index 199 onward

Delta between best epoch checkpoints:
tc202: 0.24216 at epoch index 196
tc205: 0.25296 at epoch index 195
Delta:+0.01080
Delta percentage pts: +1.08 pp


## 2. Computation of Latex table values
- this computes the values for the Latex Values
- The globals define tcs that are integrated in the comparison and which tc is used as the reference baseline

In [21]:
def get_tc_dirs(tc_min, tc_max):
    rows = []
    for config in RUNS_ROOT.rglob("config.json"):
        run_dir = config.parent
        exp_dir = run_dir.parent
        exp_name = exp_dir.name
        tc = get_current_tc(exp_name)

        if tc is None or not (tc_min <= tc <= tc_max):
            continue

        metrics_path = run_dir / "metrics.json"
        with open(config) as f:
            config_file = json.load(f)
        with open(metrics_path) as f:
            metrics_file = json.load(f)

        cfg = config_file["config"]
        use_repa = bool(cfg.get("use_repa", False))

        best_miou = metrics_file.get("best_miou")
        best_epoch = metrics_file.get("best_epoch")

        epochs = metrics_file.get("epochs", [])

        if best_miou is None:
            best_entry = max(epochs, key=lambda x: x["mean_iou"])
            best_miou = best_entry["mean_iou"]
            best_epoch = best_entry["epoch"]

        rows.append({
            "tc": tc,
            "experiment": exp_name,
            "run": run_dir.name,
            "use_repa": use_repa,
            "student_layer": cfg.get("repa_student_layer") if use_repa else None,
            "lambda": cfg.get("repa_lambda") if use_repa else None,
            "best_epoch": best_epoch,
            "best_miou": float(best_miou),
        })

    return pd.DataFrame(rows).sort_values( ["tc", "experiment", "run"]).reset_index(drop=True)

In [22]:
def print_table(df, baseline_tc):
    baseline_rows = df[df["tc"] == baseline_tc]
    baseline_miou = baseline_rows["best_miou"].item()

    df["delta_miou"] = df["best_miou"] - baseline_miou
    df["miou_percent"] = 100 * df["best_miou"]
    df["delta"] = 100 * df["delta_miou"]

    table = df[["tc", "experiment", "use_repa", "student_layer", "lambda","miou_percent","delta", "best_epoch"]].copy()

    table["student_layer"] = table["student_layer"].apply(
        lambda x: "--" if pd.isna(x) else str(int(x))
    )
    table["lambda"] = table["lambda"].apply(
        lambda x: "--" if pd.isna(x) else f"{x:.2f}"
    )
    table["mIoU"] = table["miou_percent"].apply(
        lambda x: "--" if pd.isna(x) else f"{x:.2f}"
    )
    table["delta"] = table["delta"].apply(
        lambda x: "--" if pd.isna(x) else f"{x:.2f}"
    )

    table = table[["tc",  "student_layer", "lambda", "mIoU", "best_epoch", "delta"]]

    display(table)

In [23]:
def print_latex_table(df, baseline_tc, student):
    baseline_rows = df[df["tc"] == baseline_tc]
    baseline_miou = baseline_rows["best_miou"].iloc[0]

    latex_table = df.copy()

    # Compute delta 
    latex_table["delta"] = ( latex_table["best_miou"] - baseline_miou) * 100
    latex_table["miou_percent"] = (latex_table["best_miou"] * 100)

    # baseline first and sort REPA runs by student layer
    latex_table["_sort_layer"] = latex_table["student_layer"].fillna(-999)
    latex_table = latex_table.sort_values(
        by=["use_repa", "_sort_layer"]
    )

    for _, row in latex_table.iterrows():
        if row["tc"] == baseline_tc:
            model = student
            layer_latex = "--"
            lambda_latex = "--"
            delta_latex = "--"

        else:
            model = f"{student}" + r"+ \repaseg{}"
            layer = row["student_layer"]

            if layer == -1: layer = 12

            layer_latex = f"${int(layer)}$"
            lambda_latex = f"${row['lambda']:.2f}$"
            delta_latex = f"${row['delta']:+.2f}$"

        miou_latex = f"${row['miou_percent']:.2f}$"

        print(
            f"{model} & "
            f"{layer_latex} & "
            f"{lambda_latex} & "
            f"{miou_latex} & "
            f"{delta_latex} \\\\"
        )

In [24]:
TC_MIN = 20
TC_MAX = 31
BASELINE_TC=20

df_scratch_config = get_tc_dirs(tc_min=TC_MIN, tc_max=TC_MAX)

print("Scratch ViT-14/S — REPA Configuration runs")
print_table(df_scratch_config, BASELINE_TC)

print_latex_table(df_scratch_config, BASELINE_TC, "ViT-S/14")

Scratch ViT-14/S — REPA Configuration runs


,tc,student_layer,lambda,mIoU,best_epoch,delta
0,20,--,--,21.95,118,0.00
1,21,6,0.50,24.11,116,2.16
2,22,-1,0.50,22.88,118,0.94
3,23,4,0.50,24.75,117,2.80
4,24,9,0.50,24.34,111,2.39
5,25,2,0.50,25.14,116,3.19
6,26,4,0.25,24.16,119,2.22
7,27,4,0.75,24.77,111,2.82
8,28,4,1.00,25.11,117,3.16
9,29,2,0.25,24.41,118,2.46


ViT-S/14 & -- & -- & $21.95$ & -- \\
ViT-S/14+ \repaseg{} & $12$ & $0.50$ & $22.88$ & $+0.94$ \\
ViT-S/14+ \repaseg{} & $2$ & $0.50$ & $25.14$ & $+3.19$ \\
ViT-S/14+ \repaseg{} & $2$ & $0.25$ & $24.41$ & $+2.46$ \\
ViT-S/14+ \repaseg{} & $2$ & $0.75$ & $25.62$ & $+3.67$ \\
ViT-S/14+ \repaseg{} & $2$ & $1.00$ & $25.78$ & $+3.83$ \\
ViT-S/14+ \repaseg{} & $4$ & $0.50$ & $24.75$ & $+2.80$ \\
ViT-S/14+ \repaseg{} & $4$ & $0.25$ & $24.16$ & $+2.22$ \\
ViT-S/14+ \repaseg{} & $4$ & $0.75$ & $24.77$ & $+2.82$ \\
ViT-S/14+ \repaseg{} & $4$ & $1.00$ & $25.11$ & $+3.16$ \\
ViT-S/14+ \repaseg{} & $6$ & $0.50$ & $24.11$ & $+2.16$ \\
ViT-S/14+ \repaseg{} & $9$ & $0.50$ & $24.34$ & $+2.39$ \\


In [25]:
TC_MIN = 202
TC_MAX = 205
BASELINE_TC=202

df_scratch_main = get_tc_dirs(tc_min=TC_MIN, tc_max=TC_MAX)

print("Scratch ViT-14/S — MAin Comparison ")
print_table(df_scratch_main, BASELINE_TC)

print_latex_table(df_scratch_main, BASELINE_TC,  "ViT-S/14")

Scratch ViT-14/S — MAin Comparison 


,tc,student_layer,lambda,mIoU,best_epoch,delta
0,202,--,--,24.22,194,0.00
1,203,2,1.00,27.31,199,3.09
2,205,2,1.00,25.30,193,1.08


ViT-S/14 & -- & -- & $24.22$ & -- \\
ViT-S/14+ \repaseg{} & $2$ & $1.00$ & $27.31$ & $+3.09$ \\
ViT-S/14+ \repaseg{} & $2$ & $1.00$ & $25.30$ & $+1.08$ \\


In [26]:
TC_MIN = 70
TC_MAX = 75
BASELINE_TC=70

df_pretrained = get_tc_dirs(tc_min=TC_MIN, tc_max=TC_MAX)

print("Pretrained AugReg-ViT-16/S")
print_table(df_pretrained, BASELINE_TC)

print_latex_table(df_pretrained, BASELINE_TC,  "ViT-S/16")

Pretrained AugReg-ViT-16/S


,tc,student_layer,lambda,mIoU,best_epoch,delta
0,70,--,--,46.68,79,0.00
1,71,2,1.00,45.31,73,-1.38
2,72,6,0.50,46.16,79,-0.52
3,73,2,0.50,45.33,79,-1.35
4,74,9,0.50,46.03,78,-0.66
5,75,6,0.25,46.06,76,-0.62


ViT-S/16 & -- & -- & $46.68$ & -- \\
ViT-S/16+ \repaseg{} & $2$ & $1.00$ & $45.31$ & $-1.38$ \\
ViT-S/16+ \repaseg{} & $2$ & $0.50$ & $45.33$ & $-1.35$ \\
ViT-S/16+ \repaseg{} & $6$ & $0.50$ & $46.16$ & $-0.52$ \\
ViT-S/16+ \repaseg{} & $6$ & $0.25$ & $46.06$ & $-0.62$ \\
ViT-S/16+ \repaseg{} & $9$ & $0.50$ & $46.03$ & $-0.66$ \\


## 3. Create final mIoU curve for thesis

In [27]:
from matplotlib.ticker import FuncFormatter, MultipleLocator
from matplotlib.lines import Line2D

def epoch_to_iteration(epoch, steps_per_epoch=1264):
    return epoch * steps_per_epoch

def iteration_to_epoch(iteration, steps_per_epoch=1264):
    return iteration / steps_per_epoch

def save_miou_figure(repa_path, repa, baseline, repa_tc, baseline_tc, teacher, save_dir):
    with open(repa_path.parent / "config.json") as f:
        repa_config = json.load(f)["config"]

        student_layer= int(repa_config["repa_student_layer"])
        student_layer = 12 if student_layer == -1 else student_layer
        repa_lambda = float(repa_config["repa_lambda"])

        fig, ax = plt.subplots(figsize=(6, 4.2)) # originally (6, 3.8)

        best_base_idx = baseline["miou"].idxmax()
        best_repa_idx = repa["miou"].idxmax()

        best_base_epoch = int(baseline.loc[best_base_idx, "epoch"])
        best_repa_epoch = int(repa.loc[best_repa_idx, "epoch"])
        best_base_miou = float(baseline.loc[best_base_idx, "miou"]) *100
        best_repa_miou = float(repa.loc[best_repa_idx, "miou"]) *100

        # line_base, = ax.plot(baseline["epoch"], baseline["miou"], marker="o", markevery=(9, 10), markersize=2.8, color="#54D3B3", linewidth = "1")
        # line_repa, = ax.plot(repa["epoch"], repa["miou"], marker="o",  markevery=(9, 10), markersize=2.8, color="#9C49B8", linewidth = "1")

        line_base, = ax.plot(baseline["epoch"], baseline["miou"] *100, marker="o", markevery=(9, 10), markersize=3.6, color="#4C78A8", linewidth = "1.5")
        line_repa, = ax.plot(repa["epoch"], repa["miou"]*100, marker="o",  markevery=(9, 10), markersize=3.6, color="#E45756", linewidth = "1.5")

        # set top iteration axis
        secax = ax.secondary_xaxis(
            "top",
            functions=(epoch_to_iteration, iteration_to_epoch),
        )
        secax.xaxis.set_major_locator(MultipleLocator(50_000)) # originally 25_000
        secax.xaxis.set_major_formatter(
            FuncFormatter(lambda x, pos: "0" if x == 0 else f"{x/1000:g}K")
        )
        secax.set_xlabel("Training iteration")

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation mIoU")

        ax.set_axisbelow(True)
        ax.grid(True, which="major", color="0.9", linewidth=0.8)
        dummy = Line2D([], [], linestyle="none")

        legend = ax.legend(
            [line_base, dummy, line_repa, dummy],
            [
                "Scratch ViT-S/14",
                f"Best val. mIoU: {best_base_miou:.2f} at epoch {best_base_epoch}",
                f"Scratch ViT-S/14 + REPA-Seg ({teacher} Teacher)",
                f"Best val. mIoU: {best_repa_miou:.2f} at epoch {best_repa_epoch}",
            ],
            loc="lower right",
            fontsize=9,
            handlelength=2.5,
        )

        legend.get_texts()[1].set_fontsize(7)
        legend.get_texts()[1].set_color("0.4")
        legend.get_texts()[3].set_fontsize(7)
        legend.get_texts()[3].set_color("0.4")

        ymin, ymax = ax.get_ylim()
        ax.set_ylim(ymin, ymax + 0.08 * (ymax - ymin))

        fig.tight_layout()

        save_dir = save_dir / f"miou_tc{baseline_tc}_tc{repa_tc}_sl{student_layer}_{f"{round(repa_lambda * 100):03d}"}.png"
        fig.savefig(save_dir, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"Saved miou figure in {save_dir}")

In [28]:
# Save miou per epoch figure for tc202 and tc203
BASELINE_TC = 202
REPA_TC = 203
TEACHER = "DINOv2-B/14"

SAVE_DIR = Path(f"{REPO_ROOT}/analysis/results")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

STEPS_PER_EPOCH = 1264

baseline, baseline_path = load_miou(BASELINE_TC)
repa, repa_path = load_miou(REPA_TC)

save_miou_figure(repa_path=repa_path, repa=repa, baseline=baseline, repa_tc=REPA_TC, baseline_tc=BASELINE_TC, teacher=TEACHER, save_dir=SAVE_DIR)

Saved miou figure in /home/h/haerlea/REPASeg/analysis/results/miou_tc202_tc203_sl2_100.png


In [29]:
# Save miou per epoch figure for tc202 and tc205
BASELINE_TC = 202
REPA_TC = 205
TEACHER = "MAE-B/16"

SAVE_DIR = Path(f"{REPO_ROOT}/analysis/results")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

STEPS_PER_EPOCH = 1264

baseline, baseline_path = load_miou(BASELINE_TC)
repa, repa_path = load_miou(REPA_TC)

save_miou_figure(repa_path=repa_path, repa=repa, baseline= baseline, repa_tc=REPA_TC, baseline_tc=BASELINE_TC, teacher=TEACHER, save_dir=SAVE_DIR)

Saved miou figure in /home/h/haerlea/REPASeg/analysis/results/miou_tc202_tc205_sl2_100.png
